### Preprocessing

In [1]:
import librosa
import numpy as np

SAMPLE_RATE = 22050
DURATION = 5
SAMPLES = SAMPLE_RATE * DURATION
N_MELS = 128

def preprocess_audio(file_path):

    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    # Fix length
    if len(signal) > SAMPLES:
        signal = signal[:SAMPLES]
    else:
        signal = np.pad(signal, (0, SAMPLES - len(signal)))

    # Mel Spectrogram
    mel = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Normalize
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

    return mel_db

### Load Dataset

In [2]:
import os

CLASS_NAMES = ["asthma", "copd", "Bronchial", "pneumonia", "healthy"]

def load_dataset(path):

    X, y = [], []

    for label, cls in enumerate(CLASS_NAMES):
        folder = os.path.join(path, cls)

        for file in os.listdir(folder):
            if file.endswith(".wav"):
                try:
                    feature = preprocess_audio(os.path.join(folder, file))
                    X.append(feature)
                    y.append(label)
                except:
                    print("Error loading:", file)

    return np.array(X), np.array(y)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
print("Current working directory:", os.getcwd())
print("Files in current directory:", os.listdir("."))

Current working directory: /content
Files in current directory: ['.config', 'drive', 'sample_data']


In [5]:

DATASET_PATH = "/content/drive/MyDrive/datasets/Audio_Detection_Dataset/"
X, y = load_dataset(DATASET_PATH)

X = X[..., np.newaxis]  # add channel




### Train/Test Split

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y
)

### Handle Class Imbalance

In [7]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    "balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

### Model Code

In [15]:

import tensorflow as tf
from tensorflow.keras import layers, models

def build_model():

    input_layer = layers.Input(shape=(128, 216, 1))

    # CNN
    x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(input_layer)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2,2))(x)

    # reshape for LSTM
    x = layers.Reshape((x.shape[1], x.shape[2] * x.shape[3]))(x)

    # BiLSTM
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=False))(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    output = layers.Dense(5, activation='softmax')(x)

    model = models.Model(inputs=input_layer, outputs=output)

    return model


# Build model
model = build_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

callbacks = [

    tf.keras.callbacks.EarlyStopping(
        patience=10,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        factor=0.3,
        patience=4,
        min_lr=1e-6
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks
)

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 128, 216, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 128, 216, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 64, 108, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 64, 108, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 32, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 32, 54, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 16, 27, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_4 (Reshape)             │ (None, 16, 3456)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 128)            │     1,802,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,912,581 (7.30 MB)

 Trainable params: 1,912,581 (7.30 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.2500 - loss: 1.6106 - val_accuracy: 0.3909 - val_loss: 1.5716 - learning_rate: 3.0000e-04
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.3440 - loss: 1.5067 - val_accuracy: 0.3292 - val_loss: 1.4175 - learning_rate: 3.0000e-04
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.3388 - loss: 1.4438 - val_accuracy: 0.3868 - val_loss: 1.3743 - learning_rate: 3.0000e-04
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.3554 - loss: 1.4242 - val_accuracy: 0.3992 - val_loss: 1.3217 - learning_rate: 3.0000e-04
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.3729 - loss: 1.4220 - val_accuracy: 0.3909 - val_loss: 1.3417 - learning_rate: 3.0000e-04
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.3791 - loss: 1.3759 - val_accuracy: 0.3827 - val_loss: 1.3081 - learning_rate: 3.0000e-04
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.4318 

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(
        patience=8,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "best_model.h5",
        save_best_only=True
    )
]

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks
)

In [8]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred = np.argmax(model.predict(X_val), axis=1)

print(confusion_matrix(y_val, y_pred))

print(classification_report(
    y_val,
    y_pred,
    target_names=CLASS_NAMES
))

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
[[49  5  0  4  0]
 [12 55  5  6  2]
 [ 2  3 15  0  1]
 [ 9  2  3 40  3]
 [ 0  2  5  4 16]]
              precision    recall  f1-score   support

      asthma       0.68      0.84      0.75        58
        copd       0.82      0.69      0.75        80
   Bronchial       0.54      0.71      0.61        21
   pneumonia       0.74      0.70      0.72        57
     healthy       0.73      0.59      0.65        27

    accuracy                           0.72       243
   macro avg       0.70      0.71      0.70       243
weighted avg       0.73      0.72      0.72       243

